# Lab 9 · Mapa causal de calidad de servicio y abandono
### Tema 9 · MU Gestión de Sistemas Complejos · Caso transversal: operador de telecomunicaciones

**Entorno:** SageMaker Notebook (kernel Python 3) · **Librerías:** pandas, NetworkX, matplotlib
**Duración estimada:** 150–180 min · **Nivel:** intermedio-avanzado

---

## 1. Idea general del laboratorio

El salto más difícil y valioso de la asignatura: **pasar de la correlación a la causa**. La dirección no
quiere solo saber *qué* clientes se irán, sino **por qué**, porque solo conociendo la causa puede actuar.
Reducir las reclamaciones dificultando reclamar no reduce el abandono: oculta el síntoma. La causa real
está más atrás: **inversión en red → calidad técnica → interrupciones → reclamaciones e insatisfacción →
abandono**. Construiremos un **mapa causal** (un grafo dirigido) de esa cadena. El objetivo no es *demostrar*
causalidad (eso requiere experimentos), sino **razonar causalmente con rigor** desde datos observacionales.

### Conceptos clave
- **Correlación:** dos variables se mueven juntas. **Causa:** intervenir en A cambia B. **Confusión:** A y
  B se mueven juntas porque ambas dependen de un tercero (el **confusor**).
- **DAG:** grafo dirigido **acíclico**; flecha A→B = "A causa B".

> **Nota para el vídeo.** El lab original explora en **Athena (SQL)**; aquí ejecutamos los equivalentes en
> **pandas** (con el SQL en los comentarios) para correr todo en el notebook. Dataset:
> `telco_causal_lab9.csv` (900 filas) con una **estructura causal real oculta** que hay que reconstruir.

In [ ]:
# ===========================================================================
# CELDA 1 · Dependencias
# ===========================================================================
import sys
!{sys.executable} -m pip install -q networkx matplotlib pandas
print("Dependencias listas")

In [ ]:
# ===========================================================================
# CELDA 2 · Cargar el dataset causal
# ===========================================================================
import pandas as pd, numpy as np

RUTA = '.'   # local; o 's3://<tu-bucket>/causal'
df = pd.read_csv(f'{RUTA}/telco_causal_lab9.csv')
# churn medio = proporcion de clientes que abandonan (la "tasa base").
print("Filas:", df.shape[0], "| churn medio:", round(df['churn'].mean(), 3))
df.head()

## Parte A · Explorar asociaciones (equivalente a Athena)

```sql
-- Athena:
SELECT churn, AVG(calidad_tecnica), AVG(interrupciones_30d), AVG(reclamaciones_30d),
       AVG(insatisfaccion), AVG(precio_mensual)
FROM causal GROUP BY churn;
```

In [ ]:
# ===========================================================================
# CELDA 3 · Medias por churn (¿que distingue a quien se va?)
# ===========================================================================
# groupby('churn').agg(...) calcula la media de cada variable para los que se
# quedan (churn=0) y los que se van (churn=1). round(2) para legibilidad.
medias = df.groupby('churn').agg(
    calidad=('calidad_tecnica', 'mean'),
    interrupciones=('interrupciones_30d', 'mean'),
    brecha=('brecha_velocidad_pct', 'mean'),
    reclamaciones=('reclamaciones_30d', 'mean'),
    insatisfaccion=('insatisfaccion', 'mean'),
    precio=('precio_mensual', 'mean'),
).round(2)
print(medias)

Casi todo difiere entre quienes se van y quienes se quedan. **Pero correlación no es causa.** Veamos que la
**región** (proxy de la inversión) está debajo de casi todo:

```sql
SELECT region, AVG(calidad_tecnica), AVG(interrupciones_30d), AVG(reclamaciones_30d),
       AVG(insatisfaccion), AVG(churn)*100 FROM causal GROUP BY region ORDER BY 2 DESC;
```

In [ ]:
# ===========================================================================
# CELDA 4 · Medias por region (la pista del confusor)
# ===========================================================================
# Si la region de menor calidad tiene a la vez mas interrupciones, mas
# reclamaciones y mas churn, la calidad (la inversion) es un confusor comun.
por_region = df.groupby('region').agg(
    calidad=('calidad_tecnica', 'mean'),
    interrupciones=('interrupciones_30d', 'mean'),
    reclamaciones=('reclamaciones_30d', 'mean'),
    insatisfaccion=('insatisfaccion', 'mean'),
    pct_churn=('churn', lambda s: 100 * s.mean()),   # % de abandono
).round(2).sort_values('calidad', ascending=False)
print(por_region)

## Parte B · Correlaciones cuantitativas

In [ ]:
# ===========================================================================
# CELDA 5 · Correlacion de cada variable con el churn
# ===========================================================================
# Quitamos columnas no numericas antes de correlacionar.
num = df.drop(columns=['customer_id', 'region'])
# .corr() da la matriz de correlaciones (Pearson) entre todas las variables.
corr = num.corr().round(2)
# Nos quedamos con la columna 'churn' ordenada: que se asocia mas con irse.
print("Correlacion de cada variable con el churn:")
print(corr['churn'].sort_values(ascending=False))

In [ ]:
# ===========================================================================
# CELDA 6 · Mapa de calor de la matriz de correlacion
# ===========================================================================
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
# imshow dibuja la matriz como imagen; RdBu_r = rojo(+1)...azul(-1); vmin/vmax
# fijan la escala de color a [-1, 1].
plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90)   # etiquetas de ejes
plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(label='correlacion')
plt.title('Matriz de correlacion')
plt.tight_layout(); plt.show()

> **Atención.** La mayor correlación con churn será la **insatisfacción**. Pero es el **final de la
> cadena**: actuar sobre ella (descuentos para "comprar" satisfacción) no resuelve la causa raíz (la
> calidad técnica).

## Parte C · Construir el mapa causal (DAG)

Cada flecha es una **hipótesis** que afirmamos y que idealmente habría que validar.

In [ ]:
# ===========================================================================
# CELDA 7 · Definir y dibujar el DAG de hipotesis causales
# ===========================================================================
import networkx as nx

# DiGraph = grafo DIRIGIDO (las flechas tienen sentido: causa -> efecto).
D = nx.DiGraph()
# Cada par (A, B) es la hipotesis "A causa B".
relaciones = [
    ("inversion_red", "calidad_tecnica"),
    ("calidad_tecnica", "interrupciones"),
    ("calidad_tecnica", "brecha_velocidad"),
    ("interrupciones", "reclamaciones"),
    ("interrupciones", "tiempo_resolucion"),
    ("reclamaciones", "insatisfaccion"),
    ("tiempo_resolucion", "insatisfaccion"),
    ("insatisfaccion", "churn"),
    ("precio", "churn"),
]
D.add_edges_from(relaciones)

plt.figure(figsize=(12, 7))
pos = nx.spring_layout(D, seed=7, k=1.2)   # disposicion reproducible
# nx.draw dibuja nodos + flechas; arrowsize controla el tamano de la punta.
nx.draw(D, pos, with_labels=True, node_color="#aed6f1",
        node_size=2600, font_size=9, arrowsize=20, edge_color="#333")
plt.title("Mapa causal hipotetico: de la inversion al abandono")
plt.axis("off"); plt.tight_layout(); plt.show()

# is_directed_acyclic_graph = True confirma que NO hay ciclos (es un DAG valido).
print("Es aciclico (DAG):", nx.is_directed_acyclic_graph(D))

**Confusores y caminos espurios.** La **calidad técnica** causa a la vez interrupciones, brecha y (a través
de ellas) reclamaciones. Por eso interrupciones y reclamaciones correlacionan: **descienden de la misma
raíz**, no porque una cause directamente a la otra. Comprobémoslo controlando por región.

In [ ]:
# ===========================================================================
# CELDA 8 · Confusor: correlacion global vs DENTRO de cada region
# ===========================================================================
# Correlacion global entre interrupciones y reclamaciones.
glob = df[['interrupciones_30d', 'reclamaciones_30d']].corr().iloc[0, 1]
print(f"Correlacion global interrupciones-reclamaciones: {glob:.2f}")
# Si controlamos por region (proxy de la inversion/calidad), parte de la
# asociacion entre variables "hermanas" se explica por la region comun.
print("Dentro de cada region:")
for reg, g in df.groupby('region'):
    c = g[['interrupciones_30d', 'reclamaciones_30d']].corr().iloc[0, 1]
    print(f"  {reg:10s}: {c:.2f}")

## Parte D · Bucles de retroalimentación

El DAG estático es acíclico, pero en el **tiempo** aparecen bucles:
- **Reforzador:** más reclamaciones → satura soporte → más tiempo de resolución → más insatisfacción → más reclamaciones.
- **Peligroso:** churn → menos ingresos → menos inversión → peor calidad → más churn.
- **Virtuoso:** invertir → mejor calidad → menos incidencias → soporte liberado → más satisfacción.

In [ ]:
# ===========================================================================
# CELDA 9 · Detectar el ciclo del sistema dinamico
# ===========================================================================
# Construimos un grafo dirigido que SI incluye la realimentacion churn->ingresos
# ->inversion para ilustrar el bucle peligroso.
B = nx.DiGraph()
B.add_edges_from([
    ("churn", "ingresos"), ("ingresos", "inversion_red"),
    ("inversion_red", "calidad_tecnica"), ("calidad_tecnica", "interrupciones"),
    ("interrupciones", "reclamaciones"), ("reclamaciones", "insatisfaccion"),
    ("insatisfaccion", "churn"),
])
# Ahora ya NO es aciclico: simple_cycles encuentra el bucle cerrado.
print("Tiene ciclos:", not nx.is_directed_acyclic_graph(B))
print("Ciclos encontrados:")
for ciclo in nx.simple_cycles(B):
    print("  ", " -> ".join(ciclo + [ciclo[0]]))

> **Atención.** El bucle "menos ingresos → menos inversión → peor calidad → más abandono" explica por qué
> **recortar en red para ahorrar** se retroalimenta negativamente y acelera el declive.

## Parte E · Cómo validar las hipótesis causales

La correlación **sugiere** hipótesis; la intervención controlada (experimento) o el cuasi-experimento son
las que permiten **afirmar** causalidad. Para cada flecha importante hay que proponer su validación
(A/B test, comparación antes/después, análisis temporal de precedencia, experimento de precios...).

## Parte F · Entregable · Parte G · Cierre

Entregable (5–7 págs.): medias por churn y por región; matriz de correlación; el **DAG** con las hipótesis;
**confusor principal** y una correlación espuria; **dos bucles** (virtuoso y peligroso); validación de tres
flechas; y la recomendación final (intervenir sobre la **causa raíz** —inversión/calidad— y no sobre la
insatisfacción). Si montaste Athena, limpia tabla/BD y vacía el bucket.

---

### Cierre didáctico
Has pasado de "qué predice el abandono" a "qué lo causa": construiste un mapa causal, identificaste el
confusor, reconociste los bucles y entendiste que la correlación solo sugiere y la intervención confirma.
El Lab 10 cierra el círculo **auditando los sesgos** de los modelos.